In [27]:
!nvidia-smi

Wed Mar 11 22:16:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P0             32W /   70W |    3123MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [28]:
import sys
print(sys.version)

try:
    import torch
    print("torch version:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    print("gpu name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
except Exception as e:
    print("torch issue:", e)

3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
torch version: 2.10.0+cu128
cuda available: True
gpu name: Tesla T4


In [29]:
!apt-get update -y
!apt-get install -y ffmpeg git

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,120 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,614 kB]
Fetched 5,990 kB in 2s (2,542 kB/s)
Readi

In [30]:
from google.colab import files
uploaded = files.upload()

Saving test.mp4 to test.mp4


In [31]:
import os
print(os.listdir("/content"))

['.config', 'test.mp4', 'DeOldify', 'input_frames', 'colorized_frames', 'sample_data']


In [32]:
from pathlib import Path
import subprocess

video_path = "/content/test.mp4"
frames_dir = "/content/input_frames"

Path(frames_dir).mkdir(parents=True, exist_ok=True)

cmd = [
    "ffmpeg",
    "-i", video_path,
    f"{frames_dir}/frame_%06d.png"
]

subprocess.run(cmd, check=True)

CompletedProcess(args=['ffmpeg', '-i', '/content/test.mp4', '/content/input_frames/frame_%06d.png'], returncode=0)

In [33]:
!find /content/input_frames -type f | wc -l
!ls /content/input_frames | head

608
frame_000001.png
frame_000002.png
frame_000003.png
frame_000004.png
frame_000005.png
frame_000006.png
frame_000007.png
frame_000008.png
frame_000009.png
frame_000010.png


In [34]:
%cd /content
!git clone https://github.com/jantic/DeOldify.git

/content
fatal: destination path 'DeOldify' already exists and is not an empty directory.


In [35]:
%cd /content/DeOldify
!pip install -r requirements-colab.txt

/content/DeOldify


In [36]:
import torch
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())

torch: 2.10.0+cu128
cuda: True


In [37]:
%cd /content/DeOldify

from deoldify import device
from deoldify.device_id import DeviceId
device.set(device=DeviceId.GPU0)

import torch
from deoldify.visualize import *
from pathlib import Path
import warnings

torch.backends.cudnn.benchmark = True
warnings.filterwarnings("ignore", category=UserWarning, message=".*?Your .*? set is empty.*?")

/content/DeOldify


In [38]:
%cd /content/DeOldify
!mkdir -p models
!wget https://data.deepai.org/deoldify/ColorizeVideo_gen.pth -O ./models/ColorizeVideo_gen.pth

/content/DeOldify
--2026-03-11 22:19:58--  https://data.deepai.org/deoldify/ColorizeVideo_gen.pth
Resolving data.deepai.org (data.deepai.org)... 143.244.50.86, 2400:52e0:1a01::993:1
Connecting to data.deepai.org (data.deepai.org)|143.244.50.86|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 874066230 (834M) [application/octet-stream]
Saving to: ‘./models/ColorizeVideo_gen.pth’

./models/ColorizeVi 100%[===================>] 833.57M  4.90MB/s    in 2m 50s  

2026-03-11 22:22:48 (4.90 MB/s) - ‘./models/ColorizeVideo_gen.pth’ saved [874066230/874066230]



In [39]:
%cd /content/DeOldify

import torch
import functools

_original_torch_load = torch.load

def patched_torch_load(*args, **kwargs):
    if "weights_only" not in kwargs:
        kwargs["weights_only"] = False
    return _original_torch_load(*args, **kwargs)

torch.load = patched_torch_load
torch.serialization.add_safe_globals([functools.partial])

/content/DeOldify


In [41]:
%cd /content/DeOldify
!mkdir -p video/source
!cp /content/test.mp4 /content/DeOldify/video/source/test.mp4
!ls /content/DeOldify/video/source

/content/DeOldify
test.aac  test.mp4


In [42]:
%cd /content/DeOldify

result_path = colorizer.colorize_from_file_name(
    file_name="test.mp4",
    render_factor=21,
    watermarked=False
)

print(result_path)

/content/DeOldify


<div><progress max="608" value="608"></progress> 100.00% [608/608 02:17&lt;00:00]</div>

INFO:root:Video created here: video/result/test.mp4


Video created here: video/result/test.mp4
video/result/test.mp4


In [43]:
from google.colab import files
files.download('/content/DeOldify/video/result/test.mp4')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>